In [ ]:
%pip install yfinance

In [ ]:
import yfinance as yf

data = yf.download(['VOO', 'MSFT', 'QQQ'], start='2020-01-01', end='2025-01-01')
prices = data['Close']
returns = prices.pct_change().dropna()

returns.corr()

/tmp/ipykernel_33534/1240193950.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(['VOO', 'MSFT', 'QQQ'], start='2020-01-01', end='2025-01-01')
[*********************100%***********************]  3 of 3 completed

Ticker      MSFT       QQQ       VOO
Ticker                              
MSFT    1.000000  0.880372  0.820043
QQQ     0.880372  1.000000  0.928516
VOO     0.820043  0.928516  1.000000

Correlation coefficient: 0.8200


In [13]:
'Close' in data.columns.levels[0]

True

In [15]:
returns.corr()

Ticker,MSFT,QQQ,VOO
Ticker,,,
MSFT,1.000000,0.880372,0.820043
QQQ,0.880372,1.000000,0.928516
VOO,0.820043,0.928516,1.000000


In [18]:
import numpy as np
import pandas as pd

# ---- sanity checks ----
needed = {'VOO', 'MSFT'}
if not needed.issubset(set(prices.columns)):
    raise ValueError(f"'prices' must have columns {needed}. Got: {list(prices.columns)}")

# ---- returns & stats ----
ret = prices[['VOO','MSFT']].pct_change().dropna()

rho = ret['VOO'].corr(ret['MSFT'])

# daily vols
sigma_voo = ret['VOO'].std()
sigma_msft = ret['MSFT'].std()

# annualized vols (optional, for reference)
sigma_voo_ann = sigma_voo * np.sqrt(252)
sigma_msft_ann = sigma_msft * np.sqrt(252)

# ---- hedge ratios (shares-per-share, based on daily stats) ----
# Hedge VOO with MSFT: how many MSFT shares offset 1 share of VOO
hr_msft_per_voo = rho * (sigma_voo / sigma_msft)

# Hedge MSFT with VOO: how many VOO shares offset 1 share of MSFT
hr_voo_per_msft = rho * (sigma_msft / sigma_voo)

print(f"Correlation ρ(VOO, MSFT): {rho:.4f}\n")

print("Daily volatility:")
print(f"  σ_VOO  (daily): {sigma_voo:.4%}")
print(f"  σ_MSFT (daily): {sigma_msft:.4%}\n")

print("Annualized volatility (≈√252):")
print(f"  σ_VOO  (annual): {sigma_voo_ann:.2%}")
print(f"  σ_MSFT (annual): {sigma_msft_ann:.2%}\n")

print("Hedge ratios (shares per share, using daily stats):")
print(f"  Hedge VOO with MSFT -> MSFT_per_VOO: {hr_msft_per_voo:.4f}")
print(f"  Hedge MSFT with VOO -> VOO_per_MSFT: {hr_voo_per_msft:.4f}")

# If you plan to hedge with options, divide these by option delta to get contracts-per-100-shares.
# e.g., msft_put_contracts_per_100_voo_shares = (hr_msft_per_voo * 1) / (100 * msft_put_delta)


Correlation ρ(VOO, MSFT): 0.8200

Daily volatility:
  σ_VOO  (daily): 1.3416%
  σ_MSFT (daily): 1.9211%

Annualized volatility (≈√252):
  σ_VOO  (annual): 21.30%
  σ_MSFT (annual): 30.50%

Hedge ratios (shares per share, using daily stats):
  Hedge VOO with MSFT -> MSFT_per_VOO: 0.5727
  Hedge MSFT with VOO -> VOO_per_MSFT: 1.1743
